# FASE 3 — Feature Engineering Avanzado

**Objetivo:** Construir y consolidar las variables predictivas estructuradas en los grupos técnicos del proyecto (características de máquina, telemetría original, estadísticas móviles, deltas de tendencia, historial de errores y mantenimiento), respetando estrictamente la consistencia temporal y previniendo el *data leakage*.

El notebook procesará el dataset analítico unificado `../data/interim/master_dataset.parquet` y generará la matriz de características procesada para Machine Learning en `../data/processed/features_dataset.parquet`.

In [3]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

## 1. Carga del Master Dataset y Re-validación del Target (Grupo G)

**Justificación:** Cargar `master_dataset.parquet` y verificar el target candidato `failure_next_24h` para confirmar la distribución de clases (1.96% positivos) y asegurar que no exista filtración de información futura.

In [4]:
# Carga del dataset unificado
master_df = pd.read_parquet("../data/interim/master_dataset.parquet")

print("=== INFORMACIÓN GENERAL DEL MASTER DATASET ===")
print(f"Dimensiones: {master_df.shape[0]:,} filas x {master_df.shape[1]} columnas")
print(f"Máquinas únicas: {master_df['machineID'].nunique()}")
print(f"Rango temporal: desde {master_df['datetime'].min()} hasta {master_df['datetime'].max()}")

# Re-verificación del Target
target_counts = master_df['failure_next_24h'].value_counts()
target_pct = master_df['failure_next_24h'].value_counts(normalize=True) * 100

print("\n=== DISTRIBUCIÓN DEL TARGET (failure_next_24h) ===")
print(f"Instancias Clase 0 (Normal): {target_counts[0]:,} ({target_pct[0]:.2f}%)")
print(f"Instancias Clase 1 (Pre-Falla): {target_counts[1]:,} ({target_pct[1]:.2f}%)")
print(f"Razón de desbalance: 1 a {int(target_counts[0]/target_counts[1])}")

=== INFORMACIÓN GENERAL DEL MASTER DATASET ===
Dimensiones: 876,100 filas x 19 columnas
Máquinas únicas: 100
Rango temporal: desde 2015-01-01 06:00:00 hasta 2016-01-01 06:00:00

=== DISTRIBUCIÓN DEL TARGET (failure_next_24h) ===
Instancias Clase 0 (Normal): 858,916 (98.04%)
Instancias Clase 1 (Pre-Falla): 17,184 (1.96%)
Razón de desbalance: 1 a 49


## 2. Grupo A — Features de Máquina

**Justificación:** Incorporar las características estáticas de cada equipo (`age` y `model`). Codificar la variable categórica `model` mediante One-Hot Encoding (`model_model2`, `model_model3`, `model_model4`) para su consumo directo en algoritmos de ML.

In [5]:
features_df = master_df.copy()

# Codificación One-Hot Encoding para 'model'
features_df = pd.get_dummies(features_df, columns=['model'], prefix='model', drop_first=True)

print("=== GRUPO A: VARIABLES DE MÁQUINA INCORPORADAS ===")
print(features_df[['machineID', 'age', 'model_model2', 'model_model3', 'model_model4']].head())

=== GRUPO A: VARIABLES DE MÁQUINA INCORPORADAS ===
   machineID  age  model_model2  model_model3  model_model4
0          1   18         False          True         False
1         53    5         False          True         False
2         99   14         False         False         False
3         12    9         False          True         False
4          6    7         False          True         False


## 3. Grupos B y C — Telemetría y Rolling Features (Ventanas de 3h, 6h y 24h)

**Justificación:** Calcular estadísticas móviles (media y desviación estándar) agrupadas por `machineID` para capturar la tendencia y la volatilidad reciente de los sensores (`volt`, `rotate`, `pressure`, `vibration`) en ventanas de 3h, 6h y 24h.

In [6]:
# Ordenamiento estricto por máquina y tiempo
features_df = features_df.sort_values(by=['machineID', 'datetime']).reset_index(drop=True)

sensor_cols = ['volt', 'rotate', 'pressure', 'vibration']
windows = [3, 6, 24]

# Calcular media y desviación estándar móvil por máquina
for window in windows:
    for col in sensor_cols:
        # Media móvil
        features_df[f'{col}_roll_mean_{window}h'] = features_df.groupby('machineID')[col].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        ).round(4)
        # Desviación estándar móvil (volatilidad)
        features_df[f'{col}_roll_std_{window}h'] = features_df.groupby('machineID')[col].transform(
            lambda x: x.rolling(window=window, min_periods=1).std().fillna(0)
        ).round(4)

print("=== GRUPOS B Y C: CREADAS ROLLING FEATURES ===")
print(f"Total de columnas tras incluir promedios y volatilidad móvil: {len(features_df.columns)}")

=== GRUPOS B Y C: CREADAS ROLLING FEATURES ===
Total de columnas tras incluir promedios y volatilidad móvil: 45


## 4. Grupo D — Tendencias y Cambios (Deltas)

**Justificación:** Calcular la aceleración o cambio instantáneo (diferencia $t - (t-1)$) para cada sensor. Esto mide fluctuaciones o cambios bruscos inmediatos en la operación de la máquina.

In [7]:
# Deltas instantáneos (diferencia de 1 hora)
for col in sensor_cols:
    features_df[f'{col}_delta'] = features_df.groupby('machineID')[col].diff().fillna(0).round(4)

delta_cols = [f'{c}_delta' for c in sensor_cols]
print("=== GRUPO D: VARIABLES DE DELTA INCORPORADAS ===")
print(features_df[['machineID', 'datetime'] + delta_cols].head())

=== GRUPO D: VARIABLES DE DELTA INCORPORADAS ===
   machineID            datetime  volt_delta  rotate_delta  pressure_delta  \
0          1 2015-01-01 06:00:00      0.0000        0.0000          0.0000   
1          1 2015-01-01 07:00:00    -13.3386      -15.7566        -17.6174   
2          1 2015-01-01 08:00:00      8.1107      124.6023        -20.2226   
3          1 2015-01-01 09:00:00     -8.5271     -181.2005         34.0107   
4          1 2015-01-01 10:00:00     -4.8528       89.2275          2.6381   

   vibration_delta  
0           0.0000  
1          -1.6737  
2          -9.2351  
3           6.9433  
4         -15.1316  


## 5. Grupos E y F — Historial de Errores y Mantenimiento

**Justificación:** Incluir y consolidar los conteos de errores acumulados (`errors_last_24h`, `errors_last_7d`, `distinct_errors_last_24h`, `time_since_last_error_h`, `has_error_recent`) y el estado de mantenimientos/componentes (`hours_since_maintenance`, `days_since_maintenance`, `maintenance_count_30d`, `time_since_last_component_replacement_h`, `has_recent_maintenance`).

Se imputan valores nulos en `time_since_last_error_h` con un valor representativo alto (`8760.0` horas = 1 año) indicando ausencia de errores previos.

In [8]:
# Imputación segura de nulos en tiempo desde último error
if 'time_since_last_error_h' in features_df.columns:
    features_df['time_since_last_error_h'] = features_df['time_since_last_error_h'].fillna(8760.0).round(2)

error_maint_cols = [
    'errors_last_24h', 'errors_last_7d', 'distinct_errors_last_24h', 
    'time_since_last_error_h', 'has_error_recent',
    'hours_since_maintenance', 'days_since_maintenance', 
    'maintenance_count_30d', 'time_since_last_component_replacement_h', 
    'has_recent_maintenance'
]

print("=== GRUPOS E Y F: RESUMEN DE VARIABLES DE ERRORES Y MANTENIMIENTO ===")
print(features_df[error_maint_cols].describe().round(2).T[['mean', 'std', 'min', '50%', 'max']])

=== GRUPOS E Y F: RESUMEN DE VARIABLES DE ERRORES Y MANTENIMIENTO ===
                                           mean      std  min     50%      max
errors_last_24h                            0.11     0.35  0.0    0.00     4.00
errors_last_7d                             0.75     0.92  0.0    1.00     7.00
distinct_errors_last_24h                   0.11     0.35  0.0    0.00     4.00
time_since_last_error_h                  432.99  1327.79  0.0  167.00  8760.00
has_error_recent                           0.09     0.29  0.0    0.00     1.00
hours_since_maintenance                  263.62   322.14  0.0  205.00  4079.00
days_since_maintenance                    10.98    13.42  0.0    8.54   169.96
maintenance_count_30d                      2.29     0.95  0.0    2.00     5.00
time_since_last_component_replacement_h  263.62   322.14  0.0  205.00  4079.00
has_recent_maintenance                     0.06     0.24  0.0    0.00     1.00


## 6. Verificación de Integridad y Ausencia de Nulos

**Justificación:** Confirmar que la matriz de características procesada no contenga valores `NaN`, inf ni duplicados antes de guardarla.

In [9]:
null_counts = features_df.isnull().sum()
total_nulls = null_counts.sum()

print("=== VERIFICACIÓN DE INTEGRIDAD DE LA MATRIZ DE FEATURES ===")
print(f"Total de valores nulos en la matriz: {total_nulls}")
print(f"Dimensiones finales: {features_df.shape[0]:,} filas x {features_df.shape[1]} columnas")
if total_nulls > 0:
    print("\nColumnas con valores nulos:")
    print(null_counts[null_counts > 0])
else:
    print("✔ La matriz está libre de valores nulos.")

=== VERIFICACIÓN DE INTEGRIDAD DE LA MATRIZ DE FEATURES ===
Total de valores nulos en la matriz: 0
Dimensiones finales: 876,100 filas x 49 columnas
✔ La matriz está libre de valores nulos.


## 7. Guardado del Dataset de Machine Learning Procesado (`features_dataset.parquet`)

**Justificación:** Exportar el dataset procesado listo para el modelado en `../data/processed/features_dataset.parquet`.

In [11]:
output_dir = '../data/processed/'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'features_dataset.parquet')

# Exportar a Parquet
features_df.to_parquet(output_path, index=False)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print("=== EXPORTACIÓN COMPLETADA EXITOSAMENTE ===")
print(f"Ruta del archivo procesado: {output_path}")
print(f"Tamaño del archivo: {file_size_mb:.2f} MB")
print(f"Matriz procesada: {features_df.shape[0]:,} filas x {features_df.shape[1]} columnas")

=== EXPORTACIÓN COMPLETADA EXITOSAMENTE ===
Ruta del archivo procesado: ../data/processed/features_dataset.parquet
Tamaño del archivo: 149.65 MB
Matriz procesada: 876,100 filas x 49 columnas
